In [ ]:
%load_ext autoreload
%autoreload 2

import polars as pl

from deephit_cancer_comparison.constants import DATA_PATH

SEER_ORIGINAL_DATA_PATH = DATA_PATH / "seer_original" / "seer_dataset.txt"
CANCER_SPECIFIC_DATA_PATH = DATA_PATH / "cancer_specific_data"

In [ ]:
SEER_MISSING_PATTERNS = [
    "Unknown",
    "NA",
    "Blank(s)",
    "unknown",
    "blank(s)",
    "blank",
    "not applicable",
    "unknown/not applicable",
    "na",
    "999",
    "9999",
    "99999",
]
seer_df = pl.read_csv(
    SEER_ORIGINAL_DATA_PATH,
    null_values=SEER_MISSING_PATTERNS,
).drop(
    pl.col("Site recode ICD-O-3 2023 Revision")  # Using the expanded version of this variable
)

In [ ]:
col_map = {
    "Race and origin recode (NHW, NHB, NHAIAN, NHAPI, Hispanic)": "race_origin",
    "Age recode with single ages and 90+": "age",
    "Combined Summary Stage with Expanded Regional Codes (2004+)": "summary_stage",
    "Histology recode - broad groupings": "histology",
    "Marital status at diagnosis": "marital_status",
    "Sequence number": "sequence_number",
    "Site recode ICD-O-3 2023 Revision Expanded": "site_recode",
    "Tumor Size Over Time Recode (1988+)": "tumor_size",
    "SEER cause-specific death classification": "cause_specific_death",
    "SEER other cause of death classification": "other_cause_death",
    "Survival months": "survival_months",
    "Survival months flag": "survival_months_flag",
    "Vital status recode (study cutoff used)": "vital_status",
    "Grade Recode (thru 2017)": "grade",
    "Year of diagnosis": "year_dx",
    "Sex": "sex",
}
seer_df = seer_df.rename(col_map)

In [ ]:
top10cancers = (seer_df.group_by("site_recode").len().sort("len", descending=True).head(10))[
    "site_recode"
].to_list()

print("=== TOP 10 MOST PREVALENT CANCERS ===")
print("\n".join(f"{cancer}" for cancer in top10cancers))

seer_df = seer_df.filter(
    pl.col("site_recode").is_in(top10cancers)
)  # Considering only top 10 most prevalent cancers

In [ ]:
print("=== SHAPE ===")
print(f"Rows: {seer_df.shape[0]:,} | Columns: {seer_df.shape[1]:,}")

In [ ]:
print("=== DTYPES ===")
print("\n".join(f"{col} | {dtype}" for col, dtype in zip(seer_df.columns, seer_df.dtypes)))

In [ ]:
n_dupes = seer_df.is_duplicated().sum()
print("=== DUPLICATES ===")
print(f"Exact duplicate rows: {n_dupes:,}")

seer_df = seer_df.unique()  # Dropping exact duplicates
print(f"After dropping duplicates: {seer_df.shape[0]:,}")

In [ ]:
print("=== MISSINGNESS AUDIT ===")
miss_rows = []
for col in seer_df.columns:
    null_count = seer_df[col].null_count()
    pct = round(100 * null_count / seer_df.shape[0], 2)
    miss_rows.append(
        {
            "column": col,
            "null_count": null_count,
            "pct_missing": pct,
        }
    )

miss_df = pl.DataFrame(miss_rows).sort("pct_missing", descending=True)
miss_df

In [ ]:
print("=== SURVIVAL MONTHS SANITY CHECK ===")
temp_df = seer_df.with_columns(pl.col("survival_months").cast(pl.Float64, strict=False))
print(f"Negative values:    {(temp_df["survival_months"] < 0).sum()}")
print(f"Zero values:        {(temp_df["survival_months"] == 0).sum()}")
print(f"Values > 300 mo:    {(temp_df["survival_months"] > 300).sum()}")
with pl.Config(set_fmt_float="full"):
    print(temp_df["survival_months"].describe())

In [ ]:
cat_cols = [
    "sex",
    "race_origin",
    "summary_stage",
    "histology",
    "marital_status",
    "sequence_number",
    "grade",
    "cause_specific_death",
    "other_cause_death",
    "vital_status",
    "survival_months_flag",
]

print("=== CATEGORICAL VALUE COUNTS ===")
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(seer_df.group_by(col).agg(pl.len().alias("count")).sort("count", descending=True))

In [ ]:
print("=== YEAR OF DIAGNOSIS DISTRIBUTION ===")
(seer_df.group_by("year_dx").agg(pl.len().alias("count")).sort("year_dx", descending=True))

In [ ]:
print("=== TUMOR SIZE DISTRIBUTION ===")
temp_df = seer_df.with_columns(pl.col("tumor_size").cast(pl.Float64, strict=False))
with pl.Config(set_fmt_float="full"):
    print(temp_df["tumor_size"].describe())

In [ ]:
print("=== CANCER TYPE COUNTS ===")
(seer_df.group_by("site_recode").agg(pl.len().alias("count")).sort("count", descending=True))

In [ ]:
def count_seer_missing(series: pl.Series) -> int:
    return (
        series.cast(pl.Utf8).str.strip_chars().str.to_lowercase().is_in(SEER_MISSING_PATTERNS).sum()
    )


print("=== MISSINGNESS PER COHORT ===")

for col in seer_df.columns:
    print(f"\n--- {col} ---")
    rows = []
    for cancer in top10cancers:
        sub = seer_df.filter(pl.col("site_recode") == cancer)
        n = sub.shape[0]
        total_miss = sub[col].null_count() + count_seer_missing(sub[col])
        pct = round(100 * total_miss / n, 2)
        rows.append({"cancer": cancer, "n": n, "missing": total_miss, "pct_missing": pct})
    print(pl.DataFrame(rows))

In [ ]:
seer_clean_df = (
    seer_df.drop(pl.col("grade"))  # Dropped due to really high missingness ratio
    .filter(
        (pl.col("survival_months").is_not_null())  # Drop rows where survival months are not known
    )
    .with_columns(
        pl.col("summary_stage").fill_null("Unknown"),  # Encode nulls as "Unknown"
        pl.col("marital_status").fill_null("Unknown"),  # Encode nulls as "Unknown"
    )
)

In [ ]:
print("=== RATIO OF DEATHS IN FIRST MONTH ===")
(
    seer_clean_df.group_by("site_recode")
    .agg(
        (pl.col("survival_months") == 0).sum().alias("zero_survival_months"),
        (pl.col("survival_months") != 0).sum().alias("not_zero_survival_months"),
    )
    .with_columns(
        (pl.col("zero_survival_months") / pl.col("not_zero_survival_months") * 100).alias(
            "pct_zero"
        )
    )
    .sort("zero_survival_months", descending=True)
)

- Use sequence number to filter patients
- Vital status column
- Survival months flag column

In [ ]:
print("=== FINAL DATASET SUMMARY ===")
print(f"Shape: {seer_clean_df.shape[0]:,} rows x {seer_clean_df.shape[1]} columns")
print(f"\nColumns: {seer_clean_df.columns}")

print("\nRemaining nulls per column:")
for col in seer_clean_df.columns:
    n = seer_clean_df[col].null_count()

    print(f"  {col}: {n:,}")

print("\nCohort sizes:")
print(seer_clean_df.group_by("site_recode").agg(pl.len().alias("n")).sort("n", descending=True))